<a href="https://colab.research.google.com/github/VintaBytes/Ciencia-de-datos/blob/main/cursos/machine-learning-python/cuadernos/cuaderno-12-guardar-y-recuperar-modelos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Guardar y recuperar un modelo entrenado en scikit-learn

En este mini-cuaderno vamos a entrenar rápidamente un modelo de clasificación y luego veremos cómo guardarlo en un archivo para poder reutilizarlo más adelante sin repetir el entrenamiento.

La parte de entrenamiento será breve, porque es un tema que ya vimos en cuadernos anteriores. En este caso, el objetivo principal es trabajar con la **persistencia del modelo**: conservar el estado aprendido durante el entrenamiento para recuperarlo en otro momento.

También veremos por qué, en un proyecto real, suele ser más conveniente guardar el **pipeline completo** y no solamente el estimador final.

## 1. Importamos las librerías necesarias

Usaremos un dataset de ejemplo incluido en `scikit-learn`, entrenaremos un modelo sencillo y luego guardaremos el modelo con `joblib`.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

import sklearn
import joblib
import sys

## 2. Cargamos un dataset de ejemplo

Usaremos el dataset Iris, que contiene mediciones de flores y la especie correspondiente. Es un problema de clasificación, porque el modelo debe predecir una categoría.

In [ ]:
# Cargamos el dataset Iris
datos = load_iris()

# Variables de entrada
X = datos.data

# Variable objetivo
y = datos.target

# Mostramos información básica
print("Cantidad de muestras:", X.shape[0])
print("Cantidad de variables:", X.shape[1])
print("Clases:", datos.target_names)

## 3. Separamos los datos en entrenamiento y prueba

Dividimos los datos para entrenar el modelo con una parte y evaluarlo con otra. Esto nos permite estimar cómo se comporta el modelo con datos que no vio durante el entrenamiento.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Datos de entrenamiento:", X_train.shape)
print("Datos de prueba:", X_test.shape)

## 4. Creamos y entrenamos el modelo

Entrenaremos un `RandomForestClassifier`. En este cuaderno no nos detendremos demasiado en el algoritmo, porque el foco está en guardar y recuperar el modelo entrenado.

In [ ]:
modelo = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

modelo.fit(X_train, y_train)

print("Modelo entrenado correctamente.")

## 5. Evaluamos rápidamente el modelo

Antes de guardar el modelo, conviene comprobar que funciona correctamente. Para eso hacemos predicciones sobre el conjunto de prueba y calculamos algunas métricas.

In [ ]:
y_pred = modelo.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)
print()
print(classification_report(y_test, y_pred, target_names=datos.target_names))

## 6. Guardamos el modelo entrenado

Hasta este punto, el modelo ya fue entrenado con los datos de entrenamiento. Eso significa que el objeto `modelo` contiene la información que aprendió durante el proceso de entrenamiento.

En algunos algoritmos esa información puede aparecer como coeficientes o parámetros internos. En otros, como árboles de decisión o Random Forest, lo aprendido incluye estructuras internas mucho más complejas.

Por eso, en `scikit-learn` solemos hablar de **guardar el modelo entrenado completo**, no solamente sus “pesos”.

Guardar el modelo nos permite reutilizarlo más adelante sin volver a ejecutar el entrenamiento. Esto puede ser importante cuando:

- el entrenamiento tarda mucho tiempo;
- el dataset es grande;
- queremos usar el modelo en otra notebook;
- queremos integrarlo en una aplicación;
- queremos conservar una versión concreta del modelo;
- queremos compartirlo dentro de un entorno controlado.

Para este ejemplo usaremos `joblib`, una herramienta habitual para guardar objetos de Scikit-learn.

Vamos a utilizar la extensión `.joblib` para que quede claro con qué mecanismo se guardó el archivo:

```python
modelo_iris.joblib
```

La extensión del archivo no “activa” ningún formato por sí sola, pero usar `.joblib` ayuda a documentar qué herramienta utilizamos para serializarlo.

### ⚠️ Advertencia de seguridad importante

**Nunca deberíamos usar `joblib.load()` con archivos de origen desconocido o no confiable.**

`joblib` utiliza mecanismos de serialización basados en `pickle`. Al cargar un archivo malicioso, la deserialización puede llegar a **ejecutar código arbitrario en la computadora**.

Por eso:

> Un archivo `.joblib` o `.pkl` debe tratarse de manera similar a un programa ejecutable: solo debemos cargarlo si confiamos en su origen.

En este cuaderno cargaremos únicamente archivos que acabamos de crear nosotros mismos.

In [ ]:
# ---------------------------------------------------------
# Guardamos el modelo entrenado
# ---------------------------------------------------------

joblib.dump(modelo, "modelo_iris.joblib")

print("Modelo guardado en el archivo modelo_iris.joblib")

## 7. Simulamos que empezamos de nuevo

Para comprobar que realmente podemos recuperar el modelo desde el archivo, vamos a eliminar la variable `modelo` de la memoria.

Esto simula una situación en la que cerramos el entorno y luego queremos volver a usar el modelo guardado.

In [ ]:
del modelo

print("La variable modelo fue eliminada de la memoria.")

## 8. Cargamos el modelo guardado

Ahora recuperamos el modelo desde el archivo `modelo_iris.joblib`.

No volveremos a entrenarlo: simplemente reconstruiremos en memoria el objeto que habíamos guardado y lo usaremos para predecir.

Recordemos la advertencia anterior:

> `joblib.load()` debe utilizarse únicamente con archivos provenientes de una fuente confiable.

En este caso sabemos que el archivo es seguro porque fue creado por nosotros unas celdas antes.

In [ ]:
# ---------------------------------------------------------
# Recuperamos el modelo guardado
# ---------------------------------------------------------

modelo_recuperado = joblib.load("modelo_iris.joblib")

print("Modelo recuperado correctamente.")

## 9. Usamos el modelo recuperado para predecir

El modelo recuperado conserva el estado que tenía después del entrenamiento, por lo que podemos usarlo directamente con `.predict()`.

Además de volver a calcular el accuracy, comprobaremos algo más interesante: si las predicciones obtenidas después de cargar el archivo son exactamente las mismas que habíamos obtenido antes de guardarlo.

In [ ]:
# ---------------------------------------------------------
# Predicciones con el modelo recuperado
# ---------------------------------------------------------

y_pred_recuperado = modelo_recuperado.predict(X_test)

accuracy_recuperado = accuracy_score(y_test, y_pred_recuperado)

print("Accuracy del modelo recuperado:", accuracy_recuperado)

# ---------------------------------------------------------
# Comprobamos si las predicciones son idénticas
# ---------------------------------------------------------

mismas_predicciones = (y_pred == y_pred_recuperado).all()

print("¿Las predicciones son exactamente iguales?:", mismas_predicciones)

## 10. Probamos el modelo con un dato nuevo

Ahora vamos a usar el modelo recuperado para clasificar una flor nueva.

El dato debe respetar exactamente la misma estructura que los datos utilizados para entrenar: cuatro valores numéricos en el mismo orden.

Usaremos valores razonables para una flor de Iris, pero distintos del ejemplo clásico que suele aparecer en este dataset.

In [ ]:
# Ejemplo de una flor nueva
# Valores: largo sépalo, ancho sépalo, largo pétalo, ancho pétalo
flor_nueva = [[5.0, 3.4, 1.5, 0.3]]

prediccion = modelo_recuperado.predict(flor_nueva)

clase_predicha = datos.target_names[prediccion[0]]

print("Clase predicha:", clase_predicha)

## 11. Otra opción: descargar el archivo

También podemos descargar el archivo `.joblib` a nuestra computadora. Esto es útil si queremos conservar una copia local.

En Google Colab esto es especialmente importante porque los archivos creados en el entorno de ejecución pueden desaparecer cuando la sesión se reinicia.

In [ ]:
from google.colab import files

files.download("modelo_iris.joblib")

## 12. Idea clave: en un proyecto real conviene guardar el pipeline completo

Hasta ahora nuestro modelo `RandomForestClassifier` recibió directamente variables numéricas y no necesitó transformaciones previas.

Pero en muchos proyectos reales existe un preprocesamiento antes del modelo:

- imputación de valores faltantes;
- escalado;
- codificación de variables categóricas;
- selección o transformación de variables.

Algunas de esas transformaciones **aprenden información de los datos de entrenamiento**.

Por ejemplo:

- un `SimpleImputer` puede aprender una mediana;
- un `StandardScaler` aprende medias y desvíos estándar;
- un `OneHotEncoder` aprende qué categorías existen.

Si guardáramos únicamente el modelo final, perderíamos esa información necesaria para preparar correctamente los datos nuevos.

Por eso, cuando el preprocesamiento forma parte del flujo de Machine Learning, normalmente conviene guardar:

> **preprocesamiento + modelo = pipeline completo**

De esta manera, al recuperar el archivo, las nuevas observaciones pasan automáticamente por exactamente las mismas transformaciones que se utilizaron durante el entrenamiento.

En el siguiente ejemplo construiremos un pipeline sencillo formado por:

```text
StandardScaler → LogisticRegression
```

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# ---------------------------------------------------------
# Creamos el pipeline completo
# ---------------------------------------------------------

pipeline = Pipeline(
    steps=[
        ("escalado", StandardScaler()),
        ("modelo", LogisticRegression(max_iter=1000))
    ]
)

# ---------------------------------------------------------
# Entrenamos el pipeline
# ---------------------------------------------------------

pipeline.fit(X_train, y_train)

# ---------------------------------------------------------
# Guardamos preprocesamiento + modelo en un solo archivo
# ---------------------------------------------------------

joblib.dump(pipeline, "pipeline_iris.joblib")

print("Pipeline completo guardado correctamente.")

## 13. Cargamos y usamos el pipeline completo

Cuando recuperamos el pipeline, no necesitamos reconstruir manualmente el escalado.

Podemos entregarle directamente los valores originales de `flor_nueva`:

```python
flor_nueva = [[5.0, 3.4, 1.5, 0.3]]
```

El pipeline conserva el `StandardScaler` ajustado durante el entrenamiento y aplica automáticamente esa transformación antes de enviar los datos a la regresión logística.

Esta es una de las ventajas principales de guardar el pipeline completo:

> no tenemos que recordar ni reproducir manualmente el preprocesamiento cada vez que queremos usar el modelo.

In [ ]:
# ---------------------------------------------------------
# Recuperamos el pipeline completo
# ---------------------------------------------------------

pipeline_recuperado = joblib.load("pipeline_iris.joblib")

# ---------------------------------------------------------
# Le pasamos el dato en su escala original
# El pipeline aplica StandardScaler automáticamente
# ---------------------------------------------------------

prediccion_pipeline = pipeline_recuperado.predict(flor_nueva)

print(
    "Clase predicha por el pipeline:",
    datos.target_names[prediccion_pipeline[0]]
)

## 14. Compatibilidad de versiones y reproducibilidad

Guardar un modelo no garantiza que podamos abrirlo correctamente dentro de varios años o en cualquier computadora.

Los objetos persistidos con `joblib` dependen del entorno de Python y de las versiones de las librerías con las que fueron creados. Un modelo guardado con una versión de Scikit-learn puede no ser compatible con otra versión diferente.

Por eso, en un proyecto real conviene registrar junto con el modelo información como:

- versión de Python;
- versión de Scikit-learn;
- versión de `joblib`;
- variables utilizadas;
- fecha de entrenamiento;
- métricas obtenidas;
- versión o referencia del dataset;
- código o procedimiento utilizado para entrenarlo.

Esto forma parte de la **reproducibilidad** del proyecto: no solo queremos conservar el archivo del modelo, sino también saber cómo fue construido.

In [ ]:
# ---------------------------------------------------------
# Registramos algunas versiones del entorno
# ---------------------------------------------------------

print("Python:", sys.version.split()[0])
print("Scikit-learn:", sklearn.__version__)
print("Joblib:", joblib.__version__)

## Cierre

Guardar un modelo entrenado permite reutilizarlo sin repetir el entrenamiento. Pero en un proyecto real conviene pensar la persistencia como algo más amplio que simplemente guardar un archivo.

Las ideas principales de este cuaderno son:

- podemos guardar y recuperar modelos entrenados con `joblib`;
- al recuperar el modelo obtenemos el mismo estado aprendido y las mismas predicciones;
- si existe preprocesamiento, conviene guardar el **pipeline completo**;
- los archivos `.joblib` o `.pkl` solo deben cargarse si provienen de una fuente confiable;
- conviene conservar información sobre las versiones del entorno y sobre cómo fue entrenado el modelo.

En Google Colab, además, debemos descargar los archivos que queramos conservar, porque el entorno de ejecución puede reiniciarse y eliminar los archivos temporales.

En síntesis:

> **Guardar el modelo permite reutilizarlo; guardar el pipeline y documentar el entorno permite reproducir correctamente todo el proceso.**

---

### Autoría y atribución

Este cuaderno forma parte del material educativo desarrollado por **VintaBytes**.

📧 **Contacto:** [vintabytes@gmail.com](mailto:vintabytes@gmail.com)

🌐 **Repositorio oficial:** [github.com/VintaBytes/Ciencia-de-datos](https://github.com/VintaBytes/Ciencia-de-datos)

Se permite compartir y reutilizar este material respetando la autoría y manteniendo visible esta atribución.

---
